# Boreal structure dynamics in tiles (boreal tile): 2020

## Carbon accumulation in forests for year 2020

April 2025  
Paul Montesano, PhD

Built from `code/maap_tools/`:
+ boreal_dynamics_tile.ipynb
+ extent_report.ipynb

In [5]:
import numpy as np
import rasterio
from rasterio.mask import mask
import pandas as pd
import geopandas as gpd
from scipy import stats
import multiprocessing as mp
from functools import partial
import random
import matplotlib.pyplot as plt
from shapely.geometry import box

import os
from tqdm import tqdm
import time
import warnings
from plotnine import *
# from plotnine import (
#     ggplot, aes, geom_boxplot, geom_point, geom_errorbar,
#     theme_minimal, labs, facet_grid, facet_wrap, scale_fill_brewer,
#     theme, element_text, scale_fill_manual, scale_x_discrete, 
#     position_dodge, guide_legend, ggtitle
# )
warnings.filterwarnings("ignore")

In [6]:
import json

In [7]:
import sys
sys.path.append('/projects/code/icesat2_boreal/lib')

In [8]:
from rastoolslib import *
import mosaiclib
from CovariateUtils import write_cog

In [9]:
import matplotlib.pyplot as plt
import multiprocessing as mp
from functools import partial
import os
from tqdm import tqdm
import time

### Carbon accumulation wrapper function

In [10]:
from carbon_accumulation import *

In [8]:
# def CARBON_ACC_ANALYSIS(TILE_NUM, num_simulations = 5, random_seed = None, N_PIX_SAMPLE = 100000, DO_WRITE_COG=False,
#                                extent_type = 'tile', output_dir = "/projects/my-public-bucket/carbon_accumulation_analysis_TEST"):
    
#     # Define output directory
#     output_dir = os.path.join(output_dir, extent_type, f"{TILE_NUM:07}")    
#     os.makedirs(output_dir, exist_ok=True)
    
#     """
#     Main function to execute the entire workflow.
#     """
    
#     BASIN_COG_DICT = {
#         'topo_cog_fn':      get_cog_s3_path(TILE_NUM, mosaiclib.TOPO_TINDEX_FN_DICT['c2020updated_v2']),
#         'landcover_cog_fn': get_cog_s3_path(TILE_NUM, mosaiclib.LC_TINDEX_FN_DICT['c2020updated']),
#         'biomass_cog_fn':   get_cog_s3_path(TILE_NUM, mosaiclib.AGB_TINDEX_FN_DICT['2020_v2.1']),
#         'height_cog_fn':    get_cog_s3_path(TILE_NUM, mosaiclib.HT_TINDEX_FN_DICT['2020_v2.1']),
#         'standage_cog_fn':  get_cog_s3_path(TILE_NUM, mosaiclib.MISC_TINDEX_FN_DICT['FORESTAGE_BES_2020']),
#         #'extent_gdf_fn':    TILE_GDF_FN,
#         'tcc2020_cog_fn':   get_cog_s3_path(TILE_NUM, mosaiclib.MISC_TINDEX_FN_DICT['TCC_TP_2020']),
#         'tccslope_cog_fn':  get_cog_s3_path(TILE_NUM, mosaiclib.MISC_TINDEX_FN_DICT['TCCTREND_TP_2020']),
#         'tccpvalue_cog_fn': get_cog_s3_path(TILE_NUM, mosaiclib.MISC_TINDEX_FN_DICT['TCCTRENDPVAL_TP_2020']),
#         'decidpred_cog_fn': get_cog_s3_path(TILE_NUM, mosaiclib.MISC_TINDEX_FN_DICT['DECPRED_AB_2015']),
#     }

#     # Define paths to your raster files
#     file_paths = {
#         'biomass': BASIN_COG_DICT['biomass_cog_fn'],
#         'age': BASIN_COG_DICT['standage_cog_fn'],
#         'landcover': BASIN_COG_DICT['landcover_cog_fn'],
#         'topography': BASIN_COG_DICT['topo_cog_fn'],
#         'canopy_cover': BASIN_COG_DICT['tcc2020_cog_fn'],
#         'canopy_trend': BASIN_COG_DICT['tccslope_cog_fn'],
#         'trend_pvalue': BASIN_COG_DICT['tccpvalue_cog_fn'],
#         'deciduous_fraction': BASIN_COG_DICT['decidpred_cog_fn']
#     }
    
#     # Define class labels
#     age_class_labels = ['non-forest','1-20', '21-36', '37-60', '61-80', '81-100', '101-150', '>150']
#     age_cohort_labels = ['non-forest','re-growth forest','re-growth forest', 'young forest','young forest','young forest', 'mature forest', 'old-growth forest']
#     trend_class_labels = ['decline\n(strong)', 'decline\n(weak)', 'stable', 'increase\n(weak)', 'increase\n(strong)']+['no trend\navailable']
#     pvalue_class_labels = ['not sig', 'sig (p<0.05)']+['no trend\navailable']
#     deciduous_class_labels = ['conifer', 'mixed', 'deciduous']

#     # Break if there is no TCC trend data
#     if file_paths['canopy_trend'] is None:
#         print(f"Tile {TILE_NUM} doesnt have tree cover trends calc'd, skipping...")
#     else:
#         # Define paths to raster files
#         raster_paths = list(file_paths.values())
        
#         # Define pixel area in hectares (30m resolution)
#         # 30m x 30m = 900 sq. meters = 0.09 hectares
#         pixel_area_ha = 0.09
        
#         print("Reading and preparing raster data...")
#         # Read and prepare raster data
#         rasters = read_and_prepare_rasters(raster_paths)
        
#         print("Creating classification rasters...")
#         # Create classification rasters
#         rasters = create_class_rasters(rasters)
        
#         print("Performing Monte Carlo simulations for carbon accumulation...")

#         if np.count_nonzero(~np.isnan(rasters['canopy_trend'])) == 0:
#             print(f"Tile {TILE_NUM} has all NaN tree cover trend data; wont proceed with C accumulation analysis.\nExiting.")
#             return
            
#         carbon_results = monte_carlo_carbon_accumulation(rasters, num_simulations=num_simulations, random_seed=random_seed)
    
#         if DO_WRITE_COG:
#             raster_stack = np.stack([carbon_results['carbon_acc_mean'], carbon_results['carbon_acc_std'] , 
#                                      carbon_results['carbon_acc_ci_lower'], carbon_results['carbon_acc_ci_upper'], 
#                                      carbon_results['carbon_acc_pi_lower'], carbon_results['carbon_acc_pi_upper']
#                                     ])
#             raster_stack_names = ["carbon_acc_mean", "carbon_acc_std", 
#                                   "carbon_acc_ci_lower", "carbon_acc_ci_upper", 
#                                   "carbon_acc_pi_lower", "carbon_acc_pi_upper"]
                    
#             # write COG to disk
#             write_cog(
#                         raster_stack, 
#                         os.path.join(output_dir, f'boreal_cacc_2020_{TILE_NUM:07}.tif'), 
#                         rasters['crs'], 
#                         rasters['transform'], 
#                         raster_stack_names, 
#                         out_crs=rasters['crs'],
#                         input_nodata_value= -9999
#                          )
        
#         print("Creating pixel-level dataframe...")
#         # Create pixel-level dataframe
#         ##pixel_df = create_pixel_dataframe(rasters, carbon_results
        
#         # Example call with vector files
#         pixel_df = create_pixel_dataframe(
#             rasters=rasters, 
#             carbon_results=carbon_results,
#             vector_files={
#                 'ecoregions': '/projects/my-public-bucket/databank/wwf_terr_ecos.gpkg',
#                 #'boreal': '/projects/my-public-bucket/databank/arc/wwf_circumboreal_Dissolve.gpkg'
#             },
#             vector_columns={
#                 'ecoregions': ['ECO_NAME', 'REALM'],
#                 #'boreal': ['REALM']
#             },
#             transform=rasters['transform'],  # Get this from your open raster file
#             crs=rasters['crs']  # Get this from your open raster file
#         )
#         pixel_df['ID'] = TILE_NUM
    
#         if N_PIX_SAMPLE is None:
#             print(f"Setting sample size to full length of data frame ({len(pixel_df)})...")
#             sample_size = len(pixel_df)
#         else:
#             print(f"Setting sample size to a specific number of rows from the data frame ({N_PIX_SAMPLE})...")
#             # Save pixel dataframe (sample to avoid very large files)
#             sample_size = min(N_PIX_SAMPLE, len(pixel_df))
            
#         pixel_df.sample(sample_size).to_csv(os.path.join(output_dir, f"pixel_data_sample_{TILE_NUM:07}.csv"), index=False)
#         print(f"Saved sample of {sample_size} pixels to pixel_data_sample_{TILE_NUM:07}.csv")
    
#         # Sample 10% of the rows
#         SAMP_FRAC=0.1
#         sampled_df = pixel_df.sample(frac=SAMP_FRAC, random_state=random_seed)
#         sampled_df.to_csv(os.path.join(output_dir, f"pixel_data_sample10pct_{TILE_NUM:07}.csv"), index=False)
#         print(f"Additionally, saved sample of {SAMP_FRAC*100}% of pixels to pixel_data_sample{int(SAMP_FRAC*100)}pct_{TILE_NUM:07}.csv")
        
#         print("Calculating area and total carbon by class...")
#         # Calculate area and total carbon by class
#         summary_df = calculate_area_and_total_carbon(pixel_df, pixel_area_ha, 
#                                                     groupby_cols = ['ecoregions_REALM','ecoregions_ECO_NAME', 'age_class','age_cohort','trend_class', 'pvalue_class', 'deciduous_class']) 
#         summary_df = summary_df[summary_df['pixel_count']>0]
#         summary_df.to_csv(os.path.join(output_dir, f"summary_by_class_{TILE_NUM:07}.csv"), index=False)
        
#         print("Calculating Monte Carlo totals for carbon by class...")
#         # Get unique class combinations
#         class_combinations = []
#         for age_idx, _ in enumerate(age_class_labels):  # 8 age classes
#             for trend_idx, _ in enumerate(trend_class_labels):  # 5 trend classes
#                 for pval_idx, _ in enumerate(pvalue_class_labels):  # 2 p-value classes
#                     for decid_idx, _ in enumerate(deciduous_class_labels):  # 3 deciduous classes
#                         class_combinations.append((age_idx, trend_idx, pval_idx, decid_idx))
        
#         # Calculate Monte Carlo totals for carbon by class
#         monte_carlo_totals = calculate_monte_carlo_totals(
#             rasters, carbon_results, class_combinations, pixel_area_ha, num_simulations
#         )
#         monte_carlo_totals.to_csv(os.path.join(output_dir, f"monte_carlo_totals_{TILE_NUM:07}.csv"), index=False)
        
#         if False:
#             print("Creating visualizations...")
#             # Create visualizations
#             plot_carbon_accumulation_by_age(pixel_df, output_dir)
        
#         # Calculate grand total carbon with confidence intervals
#         grand_total_carbon = monte_carlo_totals['total_carbon_Pg_mean'].sum()
#         grand_total_ci_lower = monte_carlo_totals['total_carbon_Pg_ci_lower'].sum()
#         grand_total_ci_upper = monte_carlo_totals['total_carbon_Pg_ci_upper'].sum()
#         grand_total_pi_lower = monte_carlo_totals['total_carbon_Pg_pi_lower'].sum()
#         grand_total_pi_upper = monte_carlo_totals['total_carbon_Pg_pi_upper'].sum()
        
#         print("\n===== CARBON STOCK RESULTS =====")
#         print(f"Total Carbon Stock: {grand_total_carbon:.9f} Pg C ({grand_total_carbon/1e-9:.4f} Mg C)")
#         print(f"95% Confidence Interval: ({grand_total_ci_lower:.9f}, {grand_total_ci_upper:.9f}) Pg C")
#         print(f"95% Prediction Interval: ({grand_total_pi_lower:.9f}, {grand_total_pi_upper:.9f}) Pg C")
        
#         # Generate summary statistics by different groupings
#         print("\nGenerating summary statistics by various groupings...")
        
#         # By age class
#         age_summary = monte_carlo_totals.groupby('age_class').agg({
#             'total_carbon_Pg_mean': 'sum',
#             'total_carbon_Pg_ci_lower': 'sum',
#             'total_carbon_Pg_ci_upper': 'sum'
#         }).reset_index()
#         # Preserve categorical order
#         age_summary['age_class'] = age_summary['age_class'].astype(
#             CategoricalDtype(categories=age_class_labels, ordered=True))
#         age_summary.to_csv(os.path.join(output_dir, f"age_class_summary_{TILE_NUM:07}.csv"), index=False)
        
#         # By trend class
#         trend_summary = monte_carlo_totals.groupby('trend_class').agg({
#             'total_carbon_Pg_mean': 'sum',
#             'total_carbon_Pg_ci_lower': 'sum',
#             'total_carbon_Pg_ci_upper': 'sum'
#         }).reset_index()
#         # Preserve categorical order
#         trend_summary['trend_class'] = trend_summary['trend_class'].astype(
#             CategoricalDtype(categories=trend_class_labels, ordered=True))
#         trend_summary.to_csv(os.path.join(output_dir, f"trend_class_summary_{TILE_NUM:07}.csv"), index=False)
        
#         # By deciduous class
#         deciduous_summary = monte_carlo_totals.groupby('deciduous_class').agg({
#             'total_carbon_Pg_mean': 'sum',
#             'total_carbon_Pg_ci_lower': 'sum',
#             'total_carbon_Pg_ci_upper': 'sum'
#         }).reset_index()
#         # Preserve categorical order
#         deciduous_summary['deciduous_class'] = deciduous_summary['deciduous_class'].astype(
#             CategoricalDtype(categories=deciduous_class_labels, ordered=True))
#         deciduous_summary.to_csv(os.path.join(output_dir, f"deciduous_class_summary_{TILE_NUM:07}.csv"), index=False)
        
#         # Plot total carbon by age class with confidence intervals
#         p_total = (
#             ggplot(age_summary, aes(x='age_class', y='total_carbon_Pg_mean')) +
#             geom_bar(stat='identity', fill='steelblue', alpha=0.7) +
#             geom_errorbar(
#                 aes(ymin='total_carbon_Pg_ci_lower', ymax='total_carbon_Pg_ci_upper'),
#                 width=0.2, color='black'
#             ) +
#             labs(
#                 x='Age Class (years)',
#                 y='Total Carbon Stock (Pg C)',
#                 title='Total Carbon Stock by Forest Age Class with 95% Confidence Intervals'
#             ) +
#             theme_minimal() +
#             theme(
#                 axis_text_x=element_text(angle=45, hjust=1),
#                 figure_size=(10, 6)
#             )
#         )
#         p_total.save(os.path.join(output_dir, f'total_carbon_by_age_{TILE_NUM:07}.png'), dpi=300)
#         pixel_df = None
        
#         # Write to a JSON file
#         with open(os.path.join(output_dir, 'input_data.json'), 'w') as json_file:
#             json.dump(BASIN_COG_DICT, json_file)
            
#         print(f"\nAnalysis complete! Results saved to {output_dir}")
    


#### Helper functions
**moved to `carbon_accumulation.py`**

In [9]:
# def process_multiple_rasters(TILE_NUM_LIST, num_simulations=50, n_cores=None, RANDOM_SEED=True):
#     """
#     Process multiple rasters in parallel, combining their results.
    
#     Parameters:
#     -----------
#     TILE_NUM_LIST : list
#         List of tiles to raster files
#     num_simulations : int
#         Number of Monte Carlo simulations to run
#     n_cores : int
#         Number of cores to use (defaults to all available cores - 1)
        
#     Returns:
#     --------
#     dict
#         Dictionary with combined results
#     """
#     if n_cores is None:
#         n_cores = max(1, mp.cpu_count() - 1)
    
#     print(f"Processing {len(TILE_NUM_LIST)} tiles using {n_cores} cores...")
    
#     # Create a pool of worker processes
#     pool = mp.Pool(processes=n_cores)

#     if RANDOM_SEED:
#         print('Use random seeds...')
#         random_seeds = np.random.randint(0, 10000, size=len(TILE_NUM_LIST))
#     else:
#         print('Use the same seeds for reproducibility...')
#         random_seeds = [100 for t in TILE_NUM_LIST]
    
#     # Set up the parallel execution with different seeds for each process
#     process_func = partial(CARBON_ACC_ANALYSIS_SINGLE, num_simulations=num_simulations)
#     tasks = [(path, seed) for path, seed in zip(TILE_NUM_LIST, random_seeds)]
    
#     # Execute processing in parallel with progress bar
#     results = []
#     with tqdm(total=len(tasks), desc="Processing rasters") as pbar:
#         for task in tasks:
#             result = pool.apply_async(process_func, args=(task[0],), 
#                                      kwds={'random_seed': task[1]},
#                                      callback=lambda _: pbar.update())
#             results.append(result)
        
#         # Close the pool and wait for all processes to finish
#         pool.close()
#         pool.join()
    
#     # Collect results
#     raster_results = [res.get() for res in results]
    
#     return {
#         'raster_results': raster_results,
#         # 'combined_mean_sum': combined_mean_sum,
#         # 'combined_std': combined_std,
#         # 'combined_ci_lower': combined_ci_lower,
#         # 'combined_ci_upper': combined_ci_upper,
#         # 'combined_pi_lower': combined_pi_lower,
#         # 'combined_pi_upper': combined_pi_upper,
#         # 'all_simulation_sums': all_simulation_sums
#     }

In [10]:
# def read_and_prepare_rasters(raster_paths, nodata_values=None):
#     """
#     Read multiple raster datasets and prepare them for analysis by masking nodata values.
    
#     Parameters:
#     -----------
#     raster_paths : list
#         List of paths to raster files
#     nodata_values : dict or None
#         Dictionary mapping raster indices to their nodata values
        
#     Returns:
#     --------
#     dict
#         Dictionary with raster data and metadata
#     """
#     if nodata_values is None:
#         nodata_values = {}
    
#     rasters = {}
    
#     # Read biomass raster (index 0)
#     with rasterio.open(raster_paths[0]) as src:
#         rasters['biomass_mean'] = src.read(1)
#         rasters['biomass_std'] = src.read(2)
#         rasters['meta'] = src.meta
#         rasters['transform'] = src.transform
#         rasters['crs'] = src.crs
#         rasters['shape'] = rasters['biomass_mean'].shape
        
#         # Create a mask for valid data (not -9999)
#         mask_array = rasters['biomass_mean'] != -9999
        
#         # Apply mask to biomass data
#         rasters['biomass_mean'] = np.where(mask_array, rasters['biomass_mean'], np.nan)
#         rasters['biomass_std'] = np.where(mask_array, rasters['biomass_std'], np.nan)
    
#     # Read age raster (index 1)
#     with rasterio.open(raster_paths[1]) as src:
#         rasters['age_mean'] = src.read(1)
#         rasters['age_std'] = src.read(2)
        
#         # Apply mask to age data
#         mask_array = rasters['age_mean'] != -9999
#         rasters['age_mean'] = np.where(mask_array, rasters['age_mean'], np.nan)
#         rasters['age_std'] = np.where(mask_array, rasters['age_std'], np.nan)
    
#     # Read landcover raster (index 2)
#     with rasterio.open(raster_paths[2]) as src:
#         rasters['landcover'] = src.read(1)
#         mask_array = rasters['landcover'] != -9999
#         rasters['landcover'] = np.where(mask_array, rasters['landcover'], np.nan)
    
#     # Read topography raster (index 3)
#     with rasterio.open(raster_paths[3]) as src:
#         rasters['elevation'] = src.read(1)
#         rasters['slope'] = src.read(2)
#         rasters['tsri'] = src.read(3)
#         rasters['tpi'] = src.read(4)
#         rasters['slopemask'] = src.read(5)
        
#         # Apply mask
#         for band in ['elevation', 'slope', 'tsri', 'tpi', 'slopemask']:
#             mask_array = rasters[band] != -9999
#             rasters[band] = np.where(mask_array, rasters[band], np.nan)
            
#     if raster_paths[4] is not None:
#         # Read tree canopy cover raster (index 4)
#         with rasterio.open(raster_paths[4]) as src:
#             rasters['canopy_cover'] = src.read(1)
#             mask_array = (rasters['canopy_cover'] != -9999) & (rasters['canopy_cover'] != 255)
#             rasters['canopy_cover'] = np.where(mask_array, rasters['canopy_cover'], np.nan)
#     else:
#         rasters['canopy_cover'] = np.full_like(rasters['biomass_mean'] , -9999)

#     if raster_paths[5] is not None:
#         # Read tree canopy cover trends raster (index 5)
#         with rasterio.open(raster_paths[5]) as src:
#             rasters['canopy_trend'] = src.read(1)
#             mask_array = (rasters['canopy_trend'] != -9999) & (rasters['canopy_trend'] != 255)
#             rasters['canopy_trend'] = np.where(mask_array, rasters['canopy_trend'], np.nan)
        
#         # Read p-value raster (index 6)
#         with rasterio.open(raster_paths[6]) as src:
#             rasters['pvalue'] = src.read(1)
#             mask_array = (rasters['pvalue'] != -9999) & (rasters['pvalue'] != 255)
#             rasters['pvalue'] = np.where(mask_array, rasters['pvalue'], np.nan)
#     else:
#         rasters['canopy_trend'] = np.full_like(rasters['biomass_mean'] , -9999)
#         rasters['pvalue'] = np.full_like(rasters['biomass_mean'] , -9999)
        
#     if raster_paths[7] is not None:        
#         # Read deciduous fraction raster (index 7)
#         with rasterio.open(raster_paths[7]) as src:
#             rasters['deciduous'] = src.read(1)
#             mask_array = (rasters['deciduous'] != -9999) & (rasters['deciduous'] != 255)
#             rasters['deciduous'] = np.where(mask_array, rasters['deciduous'], np.nan)
#     else:
#         rasters['deciduous'] = np.full_like(rasters['biomass_mean'] , -9999)
        
#     return rasters

# def create_class_rasters(rasters):
#     """
#     Create classification rasters based on the input data.
    
#     Parameters:
#     -----------
#     rasters : dict
#         Dictionary with raster data
        
#     Returns:
#     --------
#     dict
#         Dictionary with added classification rasters
#     """
#     # Create age classes
#     age_classes = ['non-forest','1-20', '21-36', '37-60', '61-80', '81-100', '101-150', '>150']
#     age_bins = [0, 0, 20, 36, 60, 80, 100, 150, np.inf]

#     rasters['age_class'] = np.full_like(rasters['age_mean'], 0, dtype=np.float32) #---------------

#     for i, (lower, upper) in enumerate(zip(age_bins[:-1], age_bins[1:])):
#         mask = (rasters['age_mean'] > lower) & (rasters['age_mean'] <= upper)
#         rasters['age_class'][mask] = i
    
#     # Create canopy trend classes
#     # ['decline\n(strong)','decline\n(weak)','stable','increase\n(weak)','increase\n(strong)']
#     rasters['trend_class'] = np.full_like(rasters['canopy_trend'], np.nan, dtype=np.float32)
    
#     trend_mask_1 = rasters['canopy_trend'] < -2
#     trend_mask_2 = (rasters['canopy_trend'] >= -2) & (rasters['canopy_trend'] < -0.5)
#     trend_mask_4 = (rasters['canopy_trend'] <= 2) & (rasters['canopy_trend'] > 0.5)
#     trend_mask_5 = rasters['canopy_trend'] > 2
#     trend_mask_3 = ~(trend_mask_1 | trend_mask_2 | trend_mask_4 | trend_mask_5) & ~np.isnan(rasters['canopy_trend'])
    
#     rasters['trend_class'][trend_mask_1] = 0  # Strong decline
#     rasters['trend_class'][trend_mask_2] = 1  # Weak decline
#     rasters['trend_class'][trend_mask_3] = 2  # Stable
#     rasters['trend_class'][trend_mask_4] = 3  # Weak increase
#     rasters['trend_class'][trend_mask_5] = 4  # Strong increase
    
#     # Create p-value classes
#     # ['not sig', 'sig (p<0.05)']
#     rasters['pvalue_class'] = np.full_like(rasters['pvalue'], np.nan, dtype=np.float32)
#     rasters['pvalue_class'][rasters['pvalue'] >= 0.05] = 0  # Not significant
#     rasters['pvalue_class'][rasters['pvalue'] < 0.05] = 1   # Significant
    
#     # Create deciduous fraction classes
#     # ['conifer','mixed','deciduous']
#     rasters['deciduous_class'] = np.full_like(rasters['deciduous'], np.nan, dtype=np.float32)
#     rasters['deciduous_class'][rasters['deciduous'] < 33] = 0  # Conifer
#     rasters['deciduous_class'][(rasters['deciduous'] >= 33) & (rasters['deciduous'] <= 66)] = 1  # Mixed
#     rasters['deciduous_class'][rasters['deciduous'] > 66] = 2  # Deciduous
    
#     return rasters

# def monte_carlo_carbon_accumulation(rasters, num_simulations=50, random_seed=None):
#     """
#     Perform Monte Carlo simulations to derive carbon accumulation estimates.
    
#     Parameters:
#     -----------
#     rasters : dict
#         Dictionary with raster data
#     num_simulations : int
#         Number of Monte Carlo simulations to run
#     random_seed : int
#         Random seed for reproducibility
        
#     Returns:
#     --------
#     dict
#         Dictionary with carbon accumulation results
#     """
#     if random_seed is not None:
#         np.random.seed(random_seed)
    
#     # Initialize arrays for results
#     rows, cols = rasters['shape']
#     carbon_mean = np.full_like(rasters['biomass_mean'], np.nan)
#     carbon_std = np.full_like(rasters['biomass_std'], np.nan)
#     carbon_acc_mean = np.full_like(rasters['biomass_mean'], np.nan)
#     carbon_acc_std = np.full_like(rasters['biomass_std'], np.nan)
    
#     # Create arrays for simulation results
#     sim_carbon = np.zeros((num_simulations, rows, cols))
#     sim_age = np.zeros((num_simulations, rows, cols))
#     sim_carbon_acc = np.zeros((num_simulations, rows, cols))
    
#     # Get valid pixels (not NaN in both biomass and age)
#     # Set nan to non-forest age mean value = 0.5, std = 0
#     rasters['age_mean'] = np.nan_to_num(rasters['age_mean'], nan=0.0)  # < ---- se this to 0 or 0.5?
#     rasters['age_std'] = np.nan_to_num(rasters['age_std'], nan=0)
#     valid_mask = ~np.isnan(rasters['biomass_mean']) & ~np.isnan(rasters['age_mean'])
#     valid_indices = np.where(valid_mask)

#     # Ensure biomass standard deviation is non-negative
#     rasters['biomass_std'] = np.maximum(rasters['biomass_std'], 0)
    
#     # Ensure age standard deviation is non-negative
#     rasters['age_std'] = np.maximum(rasters['age_std'], 0)
    
#     # Perform Monte Carlo simulations
#     for i in range(num_simulations):
#         # Generate random biomass values
#         sim_biomass = np.random.normal(
#             rasters['biomass_mean'], 
#             rasters['biomass_std']        )
        
#         # Generate random age values
#         sim_age[i] = np.random.normal(
#             rasters['age_mean'], 
#             rasters['age_std']
#         )
        
#         # Calculate carbon (50% of biomass)
#         sim_carbon[i] = sim_biomass * 0.5
        
#         # Calculate carbon accumulation (carbon / age)
#         # Only for valid pixels with age > 0 to avoid division by zero
#         valid_age_mask = (sim_age[i] > 0) & valid_mask
#         sim_carbon_acc[i][valid_age_mask] = sim_carbon[i][valid_age_mask] / sim_age[i][valid_age_mask]
    
#     # Calculate mean and standard deviation of carbon and carbon accumulation
#     carbon_mean = np.mean(sim_carbon, axis=0)
#     carbon_std = np.std(sim_carbon, axis=0, ddof=1)
#     carbon_acc_mean = np.mean(sim_carbon_acc, axis=0)
#     carbon_acc_std = np.std(sim_carbon_acc, axis=0, ddof=1)
    
#     # Calculate confidence intervals for carbon accumulation
#     t_value = stats.t.ppf(0.975, num_simulations - 1)
#     carbon_acc_sem = carbon_acc_std / np.sqrt(num_simulations)
#     carbon_acc_ci_lower = carbon_acc_mean - t_value * carbon_acc_sem
#     carbon_acc_ci_upper = carbon_acc_mean + t_value * carbon_acc_sem
    
#     # Calculate prediction intervals for carbon accumulation
#     carbon_acc_pred_std = np.sqrt(carbon_acc_std**2 + carbon_acc_sem**2)
#     carbon_acc_pi_lower = carbon_acc_mean - t_value * carbon_acc_pred_std
#     carbon_acc_pi_upper = carbon_acc_mean + t_value * carbon_acc_pred_std
    
#     print('\n--Completed C accumulation analysis.--\n')
    
#     return {
#         'carbon_mean': carbon_mean,
#         'carbon_std': carbon_std,
#         'carbon_acc_mean': carbon_acc_mean,
#         'carbon_acc_std': carbon_acc_std,
#         'carbon_acc_ci_lower': carbon_acc_ci_lower,
#         'carbon_acc_ci_upper': carbon_acc_ci_upper,
#         'carbon_acc_pi_lower': carbon_acc_pi_lower,
#         'carbon_acc_pi_upper': carbon_acc_pi_upper,
#         'sim_carbon': sim_carbon,
#         'sim_carbon_acc': sim_carbon_acc
#     }

# import pandas as pd
# from pandas.api.types import CategoricalDtype

# def create_pixel_dataframe(rasters, carbon_results, 
#                            vector_files=None, vector_columns=None, transform=None, crs=None):
#     """
#     Create a pandas DataFrame with pixel-level data with ordered categorical variables,
#     and add attributes from vector files at each pixel location.
    
#     Parameters:
#     -----------
#     rasters : dict
#         Dictionary with raster data
#     carbon_results : dict
#         Dictionary with carbon accumulation results
#     vector_files : dict, optional
#         Dictionary with keys as identifiers and values as paths to geopackage files
#     vector_columns : dict, optional
#         Dictionary with keys matching vector_files and values as lists of column names to extract
#     transform : affine.Affine, optional
#         The transform of the raster data to convert pixel indices to coordinates
#     crs : str or CRS object, optional
#         The coordinate reference system of the raster data
        
#     Returns:
#     --------
#     DataFrame
#         Pandas DataFrame with pixel-level data including vector attributes
#     """
#     import numpy as np
#     import pandas as pd
#     from pandas.api.types import CategoricalDtype
#     import geopandas as gpd
#     from shapely.geometry import Point
#     import rasterio
#     from rasterio.transform import rowcol
    
#     # Get valid pixels (not NaN in important variables)
#     valid_mask = (
#         ~np.isnan(rasters['biomass_mean']) & 
#         ~np.isnan(rasters['age_mean']) &
#         ~np.isnan(carbon_results['carbon_acc_mean'])
#     )
#     valid_indices = np.where(valid_mask)
    
#     # Prepare data dictionary
#     data = {
#         'row': valid_indices[0],
#         'col': valid_indices[1],
#         'biomass': rasters['biomass_mean'][valid_indices],
#         'biomass_std': rasters['biomass_std'][valid_indices],
#         'age': rasters['age_mean'][valid_indices],
#         'age_std': rasters['age_std'][valid_indices],
#         'carbon': carbon_results['carbon_mean'][valid_indices],
#         'carbon_std': carbon_results['carbon_std'][valid_indices],
#         'carbon_acc': carbon_results['carbon_acc_mean'][valid_indices],
#         'carbon_acc_std': carbon_results['carbon_acc_std'][valid_indices],
#         'carbon_acc_ci_lower': carbon_results['carbon_acc_ci_lower'][valid_indices],
#         'carbon_acc_ci_upper': carbon_results['carbon_acc_ci_upper'][valid_indices],
#         'carbon_acc_pi_lower': carbon_results['carbon_acc_pi_lower'][valid_indices],
#         'carbon_acc_pi_upper': carbon_results['carbon_acc_pi_upper'][valid_indices]
#     }
    
#     # Add classification data
#     age_class_labels = ['non-forest','1-20', '21-36', '37-60', '61-80', '81-100', '101-150', '>150']
#     age_cohort_labels = ['non-forest','re-growth forest','re-growth forest', 'young forest','young forest','young forest', 'mature forest', 'old-growth forest']
#     trend_class_labels = ['decline\n(strong)', 'decline\n(weak)', 'stable', 'increase\n(weak)', 'increase\n(strong)']
#     pvalue_class_labels = ['not sig', 'sig (p<0.05)']
#     deciduous_class_labels = ['conifer', 'mixed', 'deciduous']
    
#     # Add class values
#     data['age_class_val'] = rasters['age_class'][valid_indices]
#     data['trend_class_val'] = rasters['trend_class'][valid_indices]
#     data['pvalue_class_val'] = rasters['pvalue_class'][valid_indices]
#     data['deciduous_class_val'] = rasters['deciduous_class'][valid_indices]
    
#     # Create DataFrame
#     df = pd.DataFrame(data)
    
#     # Add class labels using map
#     df['age_class_raw'] = df['age_class_val'].map(lambda x: age_class_labels[int(x)] if not np.isnan(x) else np.nan)
#     #df['age_cohort_raw'] = df['age_class_val'].map(lambda x: age_cohort_labels[int(x)] if not np.isnan(x) else np.nan)
#     df['trend_class_raw'] = df['trend_class_val'].map(lambda x: trend_class_labels[int(x)] if not np.isnan(x) else 'no trend\navailable') #np.nan)
#     df['pvalue_class_raw'] = df['pvalue_class_val'].map(lambda x: pvalue_class_labels[int(x)] if not np.isnan(x) else 'no trend\navailable') #np.nan)
#     df['deciduous_class_raw'] = df['deciduous_class_val'].map(lambda x: deciduous_class_labels[int(x)] if not np.isnan(x) else np.nan)
    
#     # Create ordered categorical types that preserve the order in the label lists
#     age_cat_type = CategoricalDtype(categories=age_class_labels, ordered=True)
#     #agecohort_cat_type = CategoricalDtype(categories=age_cohort_labels, ordered=True)
#     trend_cat_type = CategoricalDtype(categories=trend_class_labels+['no trend\navailable'], ordered=True)
#     pvalue_cat_type = CategoricalDtype(categories=pvalue_class_labels+['no trend\navailable'], ordered=True)
#     deciduous_cat_type = CategoricalDtype(categories=deciduous_class_labels, ordered=True)
    
#     # Convert to ordered categorical variables
#     df['age_class'] = df['age_class_raw'].astype(age_cat_type)
#     #df['age_cohort'] = df['age_cohort_raw'].astype(agecohort_cat_type)
#     df['trend_class'] = df['trend_class_raw'].astype(trend_cat_type)
#     df['pvalue_class'] = df['pvalue_class_raw'].astype(pvalue_cat_type)
#     df['deciduous_class'] = df['deciduous_class_raw'].astype(deciduous_cat_type)
    
#     # Drop the raw columns as they're no longer needed
#     df = df.drop(['age_class_raw',
#                   #'age_cohort_raw', 
#                   'trend_class_raw', 'pvalue_class_raw', 'deciduous_class_raw'], axis=1)

#     reclassification_map = dict(zip(age_class_labels, age_cohort_labels))
#     df['age_cohort'] = df['age_class'].map(reclassification_map)

#     # Process vector files if provided ------
#     if vector_files and vector_columns and transform and crs:
#         # Create a GeoDataFrame with pixel points
#         # Convert pixel indices to coordinates
#         df['x'], df['y'] = rasterio.transform.xy(transform, df['row'].values, df['col'].values)
#         geometry = [Point(x, y) for x, y in zip(df['x'], df['y'])]
#         gdf_pixels = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
        
#         # Process each vector file
#         for file_id, file_path in vector_files.items():
#             if file_id in vector_columns:
#                 # Read the vector file
#                 vector_data = gpd.read_file(file_path)
                
#                 # Make sure CRS match
#                 if vector_data.crs != gdf_pixels.crs:
#                     vector_data = vector_data.to_crs(gdf_pixels.crs)
                
#                 # Keep only the columns of interest
#                 cols_to_keep = vector_columns[file_id]
#                 vector_data = vector_data[cols_to_keep + ['geometry']]
                
#                 # Perform spatial join
#                 # We use 'sjoin' with predicate='within' to find which polygon each point falls within
#                 joined = gpd.sjoin(gdf_pixels, vector_data, how='left', predicate='within')
                
#                 # Add prefixes to avoid column name conflicts
#                 prefix = f"{file_id}_"
#                 for col in cols_to_keep:
#                     if col in joined.columns:
#                         # Transfer the joined columns to the original dataframe
#                         df[f"{prefix}{col}"] = joined[col]
        
#         # Clean up coordinate columns if not needed
#         if 'x' in df.columns and 'y' in df.columns and 'geometry' not in df.columns:
#             df = df.drop(['x', 'y'], axis=1)
    
#     return df

# def calculate_area_and_total_carbon(df, pixel_area_ha, groupby_cols = ['age_class','age_cohort','trend_class', 'pvalue_class', 'deciduous_class']):
#     """
#     Calculate area and total carbon for each class combination (summarization of data frame).
    
#     Parameters:
#     -----------
#     df : DataFrame
#         Pandas DataFrame with pixel-level data
#     pixel_area_ha : float
#         Area of a single pixel in hectares
        
#     Returns:
#     --------
#     DataFrame
#         Summary DataFrame with area and total carbon by class
#     """
#     # Group by all class combinations
#     # Count pixels and calculate area
#     summary = df.groupby(groupby_cols).size().reset_index(name='pixel_count')
#     summary['area_ha'] = summary['pixel_count'] * pixel_area_ha
    
#     # Calculate total carbon (biomass * 0.5 * area in ha) and convert to Pg (1 Pg = 10^9 Mg)
#     carbon_stats = df.groupby(groupby_cols).agg({
#         'biomass': ['mean', 'std', 'count'],
#         'age': ['mean', 'std'],
#         'carbon': ['mean', 'std', 'sum'],
#         'carbon_acc': ['mean', 'std'],
#         'carbon_acc_ci_lower': 'mean',
#         'carbon_acc_ci_upper': 'mean',
#         'carbon_acc_pi_lower': 'mean',
#         'carbon_acc_pi_upper': 'mean'
#     }).reset_index()
    
#     # Flatten multi-level columns
#     carbon_stats.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in carbon_stats.columns]
    
#     # Merge with summary
#     summary = pd.merge(summary, carbon_stats, on=groupby_cols)
    
#     # Calculate total carbon in Pg (1 Pg = 10^9 Mg)
#     #  note: this is a sum based on the mean biomass (density) of the class)
#     summary['total_carbon_Pg'] = (summary['biomass_mean'] * 0.5 * summary['area_ha'] * 1e-9)
    
#     # Preserve categorical order for plotting
#     for col in groupby_cols:
#         summary[col] = summary[col].astype(df[col].dtype)
    
#     return summary

# def calculate_monte_carlo_totals(rasters, carbon_results, class_combinations, pixel_area_ha, num_simulations=50):
#     """
#     Calculate Monte Carlo simulations for total carbon by class combination (summarization of rasters).
    
#     Parameters:
#     -----------
#     rasters : dict
#         Dictionary with raster data
#     carbon_results : dict
#         Dictionary with carbon accumulation results
#     class_combinations : list
#         List of unique class combinations
#     pixel_area_ha : float
#         Area of a single pixel in hectares
#     num_simulations : int
#         Number of Monte Carlo simulations
        
#     Returns:
#     --------
#     DataFrame
#         DataFrame with Monte Carlo simulation results for total carbon for each class combination
#     """
#     # Get simulation data
#     sim_carbon = carbon_results['sim_carbon']
    
#     # Initialize results list
#     results = []
    
#     # Extract class rasters for easier access
#     age_class = rasters['age_class']
#     trend_class = rasters['trend_class']
#     pvalue_class = rasters['pvalue_class']
#     deciduous_class = rasters['deciduous_class']
    
#     # Define class labels
#     age_class_labels = ['non-forest','1-20', '21-36', '37-60', '61-80', '81-100', '101-150', '>150']
#     trend_class_labels = ['decline\n(strong)', 'decline\n(weak)', 'stable', 'increase\n(weak)', 'increase\n(strong)']+['no trend\navailable']
#     pvalue_class_labels = ['not sig', 'sig (p<0.05)']+['no trend\navailable']
#     deciduous_class_labels = ['conifer', 'mixed', 'deciduous']
    
#     # Process each class combination
#     for age_idx, trend_idx, pval_idx, decid_idx in class_combinations:
#         # Create mask for this class combination
#         mask = (
#             (age_class == age_idx) &
#             (trend_class == trend_idx) &
#             (pvalue_class == pval_idx) &
#             (deciduous_class == decid_idx)
#         )
        
#         # Skip if no pixels in this class
#         if not np.any(mask):
#             #print(f'No pixels where age={age_class_labels[age_idx]}, trend={trend_class_labels[trend_idx]}, pval={pvalue_class_labels[pval_idx]}, decid={deciduous_class_labels[decid_idx]} ...')
#             continue
        
#         # Calculate total carbon for each simulation
#         total_carbon_sim = np.zeros(num_simulations)
#         for i in range(num_simulations):
#             # Carbon in Mg for this simulation (biomass * 0.5)
#             carbon = sim_carbon[i] * mask
#             # Total carbon in Pg (1 Pg = 10^9 Mg)
#             #total_carbon_sim[i] = np.nansum(carbon) * 0.5 * pixel_area_ha * 1e-9 # <---- check here: why * 0.5 ?? Its already converted to C
#             total_carbon_sim[i] = np.nansum(carbon) * pixel_area_ha * 1e-9
        
#         # Calculate statistics
#         mean_carbon = np.mean(total_carbon_sim)
#         std_carbon = np.std(total_carbon_sim, ddof=1)
        
#         # Calculate confidence interval
#         t_value = stats.t.ppf(0.975, num_simulations - 1)
#         sem = std_carbon / np.sqrt(num_simulations)
#         ci_lower = mean_carbon - t_value * sem
#         ci_upper = mean_carbon + t_value * sem
        
#         # Calculate prediction interval
#         pi_lower = mean_carbon - t_value * std_carbon
#         pi_upper = mean_carbon + t_value * std_carbon
        
#         # Store results
#         results.append({
#             'age_class': age_class_labels[age_idx],
#             'trend_class': trend_class_labels[trend_idx],
#             'pvalue_class': pvalue_class_labels[pval_idx],
#             'deciduous_class': deciduous_class_labels[decid_idx],
#             'total_carbon_Pg_mean': mean_carbon,
#             'total_carbon_Pg_std': std_carbon,
#             'total_carbon_Pg_ci_lower': ci_lower,
#             'total_carbon_Pg_ci_upper': ci_upper,
#             'total_carbon_Pg_pi_lower': pi_lower,
#             'total_carbon_Pg_pi_upper': pi_upper
#         })
    
#     # Convert to DataFrame
#     result_df = pd.DataFrame(results)

#     # Create ordered categorical types
#     age_cat_type = CategoricalDtype(categories=age_class_labels, ordered=True)
#     trend_cat_type = CategoricalDtype(categories=trend_class_labels, ordered=True)
#     pvalue_cat_type = CategoricalDtype(categories=pvalue_class_labels, ordered=True)
#     deciduous_cat_type = CategoricalDtype(categories=deciduous_class_labels, ordered=True)
    
#     # Convert to ordered categorical variables
#     result_df['age_class'] = result_df['age_class'].astype(age_cat_type)
#     result_df['trend_class'] = result_df['trend_class'].astype(trend_cat_type)
#     result_df['pvalue_class'] = result_df['pvalue_class'].astype(pvalue_cat_type)
#     result_df['deciduous_class'] = result_df['deciduous_class'].astype(deciduous_cat_type)
    
#     return result_df

# def plot_carbon_accumulation_by_age(df, output_dir):
#     """
#     Create boxplots of carbon accumulation by age class for different class combinations.
    
#     Parameters:
#     -----------
#     df : DataFrame
#         Pandas DataFrame with pixel-level data
#     output_dir : str
#         Directory to save output plots
        
#     Returns:
#     --------
#     None
#     """
#     os.makedirs(output_dir, exist_ok=True)
    
#     trend_colors = {
#         'decline\n(strong)': '#a6611a',
#         'decline\n(weak)': '#dfc27d',
#         'stable': '#f5f5f5',
#         'increase\n(weak)': '#80cdc1',
#         'increase\n(strong)': '#018571',
#         'no trend\navailable': 'black'
#     }
    
#     pvalue_colors = {
#         'not sig': '#fc8d59',
#         'sig (p<0.05)': '#91bfdb',
#         'no trend\navailable': 'black'
#     }
    
#     deciduous_colors = {
#         'conifer': '#5e3c99', 
#         'mixed': '#ffffbf',
#         'deciduous': '#fdb863'
#     }
    
#     # 1. Carbon accumulation by age class and trend class
#     p1 = (
#         ggplot(df, aes(x='age_class', y='carbon_acc', fill='trend_class')) +
#         geom_boxplot(position=position_dodge(width=0.8), outlier_size=1) +
#         scale_fill_manual(values=trend_colors) +
#         scale_y_continuous(limits=(0,2)) +
#         labs(
#             x='Age Class (years)',
#             y='Carbon Accumulation (Mg C/ha/year)',
#             fill='Canopy Trend'
#         ) +
#         theme_minimal() +
#         theme(
#             axis_text_x=element_text(angle=45, hjust=1),
#             figure_size=(12, 8)
#         ) +
#         ggtitle('Carbon Accumulation by Age and Canopy Trend')
#     )
#     print(p1)
#     p1.save(os.path.join(output_dir, 'carbon_acc_by_age_trend.png'), dpi=300, bg='white')
    
#     # 2. Carbon accumulation by age class and significance class
#     p2 = (
#         ggplot(df, aes(x='age_class', y='carbon_acc', fill='pvalue_class')) +
#         geom_boxplot(position=position_dodge(width=0.8), outlier_size=1) +
#         scale_fill_manual(values=pvalue_colors) +
#         scale_y_continuous(limits=(0,2)) +
#         labs(
#             x='Age Class (years)',
#             y='Carbon Accumulation (Mg C/ha/year)',
#             fill='Significance'
#         ) +
#         theme_minimal() +
#         theme(
#             axis_text_x=element_text(angle=45, hjust=1),
#             figure_size=(12, 8)
#         ) +
#         ggtitle('Carbon Accumulation by Age and Statistical Significance')
#     )
#     print(p2)
#     p2.save(os.path.join(output_dir, 'carbon_acc_by_age_significance.png'), dpi=300, bg='white')
    
#     # 3. Carbon accumulation by age class and deciduous class
#     p3 = (
#         ggplot(df, aes(x='age_class', y='carbon_acc', fill='deciduous_class')) +
#         geom_boxplot(position=position_dodge(width=0.8), outlier_size=1) +
#         scale_fill_manual(values=deciduous_colors) +
#         scale_y_continuous(limits=(0,2)) +
#         labs(
#             x='Age Class (years)',
#             y='Carbon Accumulation (Mg C/ha/year)',
#             fill='Forest Type'
#         ) +
#         theme_minimal() +
#         theme(
#             axis_text_x=element_text(angle=45, hjust=1),
#             figure_size=(12, 8)
#         ) +
#         ggtitle('Carbon Accumulation by Age and Forest Type')
#     )
#     print(p3)
#     p3.save(os.path.join(output_dir, 'carbon_acc_by_age_forest_type.png'), dpi=300, bg='white')
    
#     # 4. Carbon accumulation confidence intervals by age and trend
#     summary_df = df.groupby(['age_class', 'trend_class']).agg({
#         'carbon_acc': 'mean',
#         'carbon_acc_ci_lower': 'mean',
#         'carbon_acc_ci_upper': 'mean',
#         'carbon_acc_pi_lower': 'mean',
#         'carbon_acc_pi_upper': 'mean'
#     }).reset_index()
    
#     p4 = (
#         ggplot(summary_df, aes(x='age_class', y='carbon_acc', fill='trend_class')) +
#         geom_point(aes(color='trend_class'), size=3, position=position_dodge(width=0.8)) +
#         geom_errorbar(
#             aes(ymin='carbon_acc_ci_lower', ymax='carbon_acc_ci_upper', color='trend_class'),
#             width=0.2, position=position_dodge(width=0.8)
#         ) +
#         scale_fill_manual(values=trend_colors) +
#         scale_color_manual(values={k: v for k, v in trend_colors.items()}) +
#         scale_y_continuous(limits=(0,2)) +
#         labs(
#             x='Age Class (years)',
#             y='Carbon Accumulation (Mg C/ha/year)',
#             color='Canopy Trend',
#             fill='Canopy Trend'
#         ) +
#         theme_minimal() +
#         theme(
#             axis_text_x=element_text(angle=45, hjust=1),
#             figure_size=(12, 8)
#         ) +
#         ggtitle('Carbon Accumulation by Age and Canopy Trend (with 95% Confidence Intervals)')
#     )
#     print(p4)
#     p4.save(os.path.join(output_dir, 'carbon_acc_ci_by_age_trend.png'), dpi=300, bg='white')
    
#     # 5. Carbon accumulation prediction intervals by age and forest type
#     p5 = (
#         ggplot(summary_df, aes(x='age_class', y='carbon_acc', fill='trend_class')) +
#         geom_point(aes(color='trend_class'), size=3, position=position_dodge(width=0.8)) +
#         geom_errorbar(
#             aes(ymin='carbon_acc_pi_lower', ymax='carbon_acc_pi_upper', color='trend_class'),
#             width=0.2, position=position_dodge(width=0.8)
#         ) +
#         scale_fill_manual(values=trend_colors) +
#         scale_color_manual(values={k: v for k, v in trend_colors.items()}) +
#         scale_y_continuous(limits=(0,2)) +
#         labs(
#             x='Age Class (years)',
#             y='Carbon Accumulation (Mg C/ha/year)',
#             color='Canopy Trend',
#             fill='Canopy Trend'
#         ) +
#         theme_minimal() +
#         theme(
#             axis_text_x=element_text(angle=45, hjust=1),
#             figure_size=(12, 8)
#         ) +
#         ggtitle('Carbon Accumulation by Age and Canopy Trend (with 95% Prediction Intervals)')
#     )
#     print(p5)
#     p5.save(os.path.join(output_dir, 'carbon_acc_pi_by_age_trend.png'), dpi=300, bg='white')
    
#     # 6. Plot facet grid by all class combinations
#     # Select a subset to avoid too many plots
#     subset_df = df.sample(min(100000, len(df)))
    
#     p6 = (
#         ggplot(subset_df, aes(x='age_class', y='carbon_acc', fill='deciduous_class')) +
#         geom_boxplot(position=position_dodge(width=0.8), outlier_size=0.5) +
#         scale_fill_manual(values=deciduous_colors) +
#         facet_grid('trend_class ~ pvalue_class') +
#         scale_y_continuous(limits=(0,2)) +
#         labs(
#             x='Age Class (years)',
#             y='Carbon Accumulation (Mg C/ha/year)',
#             fill='Forest Type'
#         ) +
#         theme_minimal() +
#         theme(
#             axis_text_x=element_text(angle=90, hjust=1, size=8),
#             figure_size=(15, 12)
#         ) +
#         ggtitle('Carbon Accumulation by Age, Forest Type, Trend, and Significance')
#     )
#     print(p6)
#     p6.save(os.path.join(output_dir, 'carbon_acc_facet_grid.png'), dpi=300, bg='white')


**Used in notebook only**

In [11]:
def visualize_samples(ecoregions, sampled_tiles, output_file=None, TO_CRS=4326):
    """
    Create a map visualization of the sampled tiles and ecoregions
    
    Parameters:
    -----------
    ecoregions : gpd.GeoDataFrame
        GeoDataFrame containing ecoregions
    sampled_tiles : gpd.GeoDataFrame
        GeoDataFrame containing sampled tiles
    output_file : str, optional
        Path to save the output figure, if None, the plot is displayed
    """
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    
    # Plot ecoregions with different colors
    ecoregions.to_crs(TO_CRS).plot(ax=ax, column='ECO_NAME', legend=True, 
                    legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
                    alpha=0.5, cmap='viridis')
    
    # Plot sampled tiles
    sampled_tiles.to_crs(TO_CRS).plot(ax=ax, color='red', markersize=5, 
                       edgecolor='black', linewidth=0.5)
    
    # Add title and labels
    plt.title('Randomly Sampled Boreal Tiles by Ecoregion')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    
    # Save or display the figure
    if output_file:
        plt.savefig(output_file, bbox_inches='tight', dpi=300)
        print(f"Figure saved to {output_file}")
    else:
        plt.tight_layout()
        plt.show()



In [12]:
def sample_tiles_by_ecoregion(ecoregions_path, tiles_path, n_samples_total=None, n_samples_per_ecoregion=None, 
                              area_weighted=False, area_threshold_km2=10000, prefer_contained=True,
                              random_seed=42):
    """
    Randomly sample tiles from ecoregions that intersect a boreal tile extent and exceed an area threshold,
    preferentially selecting tiles that are completely contained within ecoregions.
    
    Parameters:
    -----------
    ecoregions_path : str
        Path to ecoregions geopackage file
    tiles_path : str
        Path to boreal tiles geopackage file
    n_samples_total : int, optional
        Total number of tiles to sample across all ecoregions (used when area_weighted=True)
    n_samples_per_ecoregion : int, optional
        Number of tiles to sample from each qualifying ecoregion (used when area_weighted=False)
    area_weighted : bool, default=False
        If True, sample tiles proportionally to ecoregion area
    area_threshold_km2 : float
        Minimum area threshold for ecoregions in square kilometers
    prefer_contained : bool, default=True
        If True, preferentially select tiles completely contained within ecoregions
    random_seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    tuple
        (sampled_tiles, intersecting_ecoregions, large_ecoregions, tiles_extent)
    """
    # Validate parameters
    if area_weighted and n_samples_total is None:
        raise ValueError("n_samples_total must be provided when area_weighted=True")
    if not area_weighted and n_samples_per_ecoregion is None:
        raise ValueError("n_samples_per_ecoregion must be provided when area_weighted=False")
    
    # Set random seed for reproducibility
    random.seed(random_seed)
    np.random.seed(random_seed)
    
    print("Reading ecoregions geopackage...")
    ecoregions = gpd.read_file(ecoregions_path)
    
    print("Reading boreal tiles geopackage...")
    tiles = gpd.read_file(tiles_path)
    
    # Ensure both datasets are in the same CRS
    if ecoregions.crs != tiles.crs:
        print(f"Reprojecting ecoregions...") # from {ecoregions.crs} to {tiles.crs}")
        ecoregions_4326 = ecoregions.to_crs(4326)
        ecoregions = ecoregions.to_crs(tiles.crs).copy()
    
    # Get the total extent of the boreal tiles
    print("Calculating boreal tiles extent...")
    tiles_extent = box(*tiles.total_bounds)
    tiles_extent_4326 = box(*tiles.to_crs(4326).total_bounds)
    
    # Select ecoregions that intersect with the boreal tiles extent
    print("Selecting ecoregions that intersect with boreal tiles extent...")
    intersecting_ecoregions = ecoregions[ecoregions.intersects(tiles_extent)]
    intersecting_ecoregions_4326 = ecoregions_4326[ecoregions_4326.intersects(tiles_extent_4326)]
    
    # Calculate area in square kilometers (assuming the CRS is in meters)
    print("Calculating ecoregion areas...")
    # Convert area to km² (if CRS is in meters)
    area_factor = 1e-6  # Convert m² to km²
    intersecting_ecoregions['area_km2'] = intersecting_ecoregions.geometry.area * area_factor

    # Subset to only the intersecting regions of the CRS 4326 total bounds
    intersecting_ecoregions = intersecting_ecoregions[intersecting_ecoregions.ECO_NAME.isin(intersecting_ecoregions_4326.ECO_NAME.to_list())]
    
    # Filter ecoregions based on area threshold
    print(f"Filtering ecoregions with area > {area_threshold_km2} km²...")
    large_ecoregions = intersecting_ecoregions[intersecting_ecoregions['area_km2'] > area_threshold_km2]
    
    print(f"Found {len(large_ecoregions)} ecoregions exceeding area threshold {area_threshold_km2} km2")
    
    # Create spatial index for tiles to speed up spatial operations
    tiles_sindex = tiles.sindex
    
    # Check each ecoregion for fully contained tiles and exclude those without any
    if prefer_contained:
        print("Checking large ecoregions for fully contained tiles...")
        list_large_ecoregions_contained_tiles = []
        list_large_ecoregions_notcontained_tiles = []
        list_large_ecoregions_notcontained_names = []
        
        for idx, ecoregion in large_ecoregions.iterrows():
            ecoregion_name = ecoregion.get('ECO_NAME', f"Ecoregion_{idx}")
            
            # Find tiles that intersect with this ecoregion
            possible_matches_idx = list(tiles_sindex.intersection(ecoregion.geometry.bounds))
            possible_matches = tiles.iloc[possible_matches_idx]
            
            # Find tiles fully contained within the ecoregion
            contained_tiles = possible_matches[possible_matches.geometry.apply(
                lambda x: ecoregion.geometry.contains(x))]
            not_contained_tiles = possible_matches[~possible_matches.geometry.apply(
                lambda x: ecoregion.geometry.contains(x))]
            
            if len(contained_tiles) > 0:
                list_large_ecoregions_contained_tiles.append(ecoregion)
            else:
                list_large_ecoregions_notcontained_tiles.append(ecoregion)
                list_large_ecoregions_notcontained_names.append(ecoregion_name)
                
        
        if list_large_ecoregions_notcontained_tiles:
            #
            # For excluded ecoregions based on prefer_contained, lets revert back 
            # to intersecting tiles and add them back in to set of valid large_ecoregions
            #
            print(f"{len(list_large_ecoregions_notcontained_tiles)} ecoregions with no fully contained tiles:")
            # for eco_name in list_large_ecoregions_notcontained_names:
            #     print(f"  - {eco_name}")
        
        # Update large_ecoregions to only include those with contained tiles
        large_ecoregions = gpd.GeoDataFrame(list_large_ecoregions_contained_tiles + list_large_ecoregions_notcontained_tiles, crs=ecoregions.crs) # <<<< ---- add list from noncontained tiles
        print(f"Proceeding with {len(large_ecoregions)} valid ecoregions")
        
        if len(large_ecoregions) == 0:
            print("No valid ecoregions remain after filtering for contained tiles.")
            return gpd.GeoDataFrame(columns=tiles.columns, crs=tiles.crs), intersecting_ecoregions, large_ecoregions, tiles_extent
    
    # Calculate number of samples per ecoregion if using area-weighted sampling
    if area_weighted:
        total_area = large_ecoregions['area_km2'].sum()
        large_ecoregions['samples_allocation'] = np.round(
            (large_ecoregions['area_km2'] / total_area) * n_samples_total
        ).astype(int)
        
        # Ensure at least one sample per ecoregion if possible
        min_samples = 1
        large_ecoregions['samples_allocation'] = np.maximum(large_ecoregions['samples_allocation'], min_samples)
        print(f"Initial sampling: {large_ecoregions['samples_allocation'].sum()} total samples based on {n_samples_total} total samples requested.")

        if False:
            # Adjust if total exceeds n_samples_total
            while large_ecoregions['samples_allocation'].sum() > n_samples_total:
                # Find ecoregions with more than minimum samples
                adjustable = large_ecoregions[large_ecoregions['samples_allocation'] > min_samples]
                
                if len(adjustable) == 0:
                    # Can't adjust anymore without going below minimum
                    break
                    
                # Reduce the count for the largest ecoregion
                idx_to_reduce = adjustable['area_km2'].idxmax()
                large_ecoregions.loc[idx_to_reduce, 'samples_allocation'] -= 1
        
        print(f"Area-weighted sampling: {large_ecoregions['samples_allocation'].sum()} total samples allocated")
        # print("Samples per ecoregion:")
        # for idx, eco in large_ecoregions.iterrows():
        #     print(f"  {eco.get('ECO_NAME', f"Ecoregion_{idx}")}: {eco['samples_allocation']} samples")
    
    # Initialize list to store sampled tiles
    sampled_tiles = []
    
    # Process each qualifying ecoregion
    for idx, ecoregion in large_ecoregions.iterrows():
        ecoregion_name = ecoregion.get('ECO_NAME', f"Ecoregion_{idx}")
        print(f"Processing ecoregion: {ecoregion_name}: {ecoregion['samples_allocation']} samples")
        
        # Find tiles that intersect with this ecoregion
        possible_matches_idx = list(tiles_sindex.intersection(ecoregion.geometry.bounds))
        possible_matches = tiles.iloc[possible_matches_idx]
        
        # Separate fully contained tiles from partially intersecting tiles
        contained_tiles = possible_matches[possible_matches.geometry.apply(
            lambda x: ecoregion.geometry.contains(x))]
        
        if not prefer_contained:
            if False: # In place of this, you want the 'not_contained_tiles' set from earlier
                # If not preferring contained tiles, use all intersecting tiles
                intersecting_tiles = possible_matches[possible_matches.intersects(ecoregion.geometry)]
                print(f"  Found {len(intersecting_tiles)} tiles intersecting this ecoregion")
            candidate_tiles = possible_matches  ## <<<<<<<--------------------------------------------- pd.concat([contained_tiles , intersecting_tiles])
            print(f"  Found {len(possible_matches)} tiles intersecting this ecoregion")
        else:
            # If preferring contained tiles, prioritize them
            print(f"  Found {len(contained_tiles)} tiles completely contained within this ecoregion")
            candidate_tiles = contained_tiles
        
        # Determine number of samples for this ecoregion
        if area_weighted:
            n_samples = min(ecoregion['samples_allocation'], len(candidate_tiles))
        else:
            n_samples = min(n_samples_per_ecoregion, len(candidate_tiles))
        
        if n_samples > 0:
            # Randomly sample n candidate tiles made from CONTAINED set
            sampled_idx = random.sample(range(len(candidate_tiles)), n_samples)
            samples = candidate_tiles.iloc[sampled_idx].copy()
            
            # Add ecoregion information to sampled tiles
            for col in large_ecoregions.columns:
                if col != 'geometry':
                    samples[f'eco_{col}'] = ecoregion[col]
            
            # Add a flag indicating if the tile is fully contained in the ecoregion
            if prefer_contained:
                samples['fully_contained'] = True
            else:
                # Check each tile if it's fully contained
                samples['fully_contained'] = samples.geometry.apply(
                    lambda x: ecoregion.geometry.contains(x))
            print(f"  Sampled {n_samples} tiles from this ecoregion")
        else:
            n_samples = min(ecoregion['samples_allocation'], len(possible_matches))
            print(f"  No CONTAINED tiles to sample from this ecoregion, so sampling {n_samples} tile(s) from all {len(possible_matches)} possible_matches...")
            # Randomly sample n candidate tiles made from INTERSECTING set
            sampled_idx = random.sample(range(len(possible_matches)), n_samples)
            samples = possible_matches.iloc[sampled_idx].copy()

        # Add to our collection
        sampled_tiles.append(samples)
    
    # Combine all sampled tiles into a single GeoDataFrame
    if sampled_tiles:
        result = gpd.GeoDataFrame(pd.concat(sampled_tiles, ignore_index=True), crs=tiles.crs)
        print(f"Successfully sampled {len(result)} tiles across {len(large_ecoregions)} ecoregions")
        return result, intersecting_ecoregions, large_ecoregions, tiles_extent
    else:
        print("No tiles were sampled.")
        return gpd.GeoDataFrame(columns=tiles.columns, crs=tiles.crs), intersecting_ecoregions, large_ecoregions, tiles_extent

## Carbon Accumulation Analysis: Loop over tiles

### `NOTE:`This works in N. America only - b/c of deciduous fraction data limits - exclude decid fraction to look at Eurasia

#### Do plotting notebook using `papermill`

In [12]:
pip install papermill

  Using cached papermill-2.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached ansicolors-1.1.8-py2.py3-none-any.whl.metadata (9.0 kB)
Using cached papermill-2.6.0-py3-none-any.whl (38 kB)
Using cached ansicolors-1.1.8-py2.py3-none-any.whl (13 kB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
import papermill as pm

### TEST analysis

In [14]:
TILE_NUM_LIST = [
                3925, 3456, 3460, # Alaska
                3109, 3201, 3110, # Alberta
                1154, 
                1100, 1045, 985, # Quebec
                2675, # Great Bear south
                2000, # south of Hudson Bay
                ]


In [15]:
DO_TEST = False

In [16]:
if DO_TEST: 
    MAINDIR = "/projects/carbon_accumulation_analysis_TEST"
    N_SIMS = 2
    SIMDIR =  f"{MAINDIR}/N_SIMS_{N_SIMS}"
    !mkdir -p $SIMDIR
    
    for TILE_NUM in TILE_NUM_LIST:
        
        if True:
            CARBON_ACC_ANALYSIS_SINGLE_v2(TILE_NUM, num_simulations = N_SIMS, N_PIX_SAMPLE=None, 
                                       random_seed = None, DO_WRITE_COG=False,
                                       extent_type = 'tile', output_dir = SIMDIR)
        
        pm.execute_notebook(
            '/projects/code/maap_tools/notebooks/boreal_dynamics_plot_v2.ipynb',
            f'{SIMDIR}/tile/{TILE_NUM:07}/boreal_dynamics_plot_v2_{TILE_NUM:07}.ipynb',
            parameters={
                'TILE_NUM ': TILE_NUM ,
                'OUTDIR': SIMDIR
            },
            kernel_name='ir'  # Specify the R kernel
        )

### Sampling tiles by ecoregion

In [12]:
# File paths
ecoregions_path = "/projects/my-public-bucket/databank/wwf_terr_ecos.gpkg"
#ecoregions2017_boreal_path = "/projects/my-public-bucket/databank/arc/Ecoregions2017_boreal_m.geojson"
# boreal_path = '/projects/my-public-bucket/databank/arc/boreal.shp'
# wwf_circdiss_path = "/projects/shared-buckets/montesano/databank/arc/wwf_circumboreal_Dissolve.gpkg"
tiles_path = "/projects/shared-buckets/montesano/databank/daac/boreal_tiles_v004_ORNLDAAC_albers.gpkg"

In [21]:
# wwf_circdiss_gdf = gpd.read_file(wwf_circdiss_path)
# boreal_gdf = gpd.read_file(boreal_path)
# eco_gdf = gpd.read_file(ecoregions2017_boreal_path)
# #boreal_gdf.explore(m=wwf_circ_gdf.explore(m=eco_gdf.explore(color='yellow'), color='red'), color='blue')
# wwf_circdiss_gdf.explore(m=eco_gdf.explore(color='yellow'), color='red')

In [22]:
# Parameters
area_threshold = 1e4  # Minimum ecoregion area in km²

# Perform sampling
sampled_tiles , int_eco, large_ecoregions, tiles_extent = sample_tiles_by_ecoregion(
    ecoregions_path, 
    tiles_path, 
    n_samples_total=250,
    n_samples_per_ecoregion=1, 
    area_threshold_km2=area_threshold, 
    area_weighted=True, 
    prefer_contained=True,
    random_seed=42
)

Reading ecoregions geopackage...
Reading boreal tiles geopackage...
Reprojecting ecoregions...
Calculating boreal tiles extent...
Selecting ecoregions that intersect with boreal tiles extent...
Calculating ecoregion areas...
Filtering ecoregions with area > 10000.0 km²...
Found 244 ecoregions exceeding area threshold 10000.0 km2
Checking large ecoregions for fully contained tiles...
186 ecoregions with no fully contained tiles:
Proceeding with 244 valid ecoregions
Initial sampling: 375 total samples based on 250 total samples requested.
Area-weighted sampling: 375 total samples allocated
Processing ecoregion: Chukchi Peninsula tundra: 1 samples
  Found 2 tiles completely contained within this ecoregion
  Sampled 1 tiles from this ecoregion
Processing ecoregion: East Siberian taiga: 21 samples
  Found 321 tiles completely contained within this ecoregion
  Sampled 21 tiles from this ecoregion
Processing ecoregion: Arctic foothills tundra: 1 samples
  Found 1 tiles completely contained wi

In [18]:
TILE_NUM_LIST = sampled_tiles.tile_num.to_list()
len(sorted(TILE_NUM_LIST))

NameError: name 'sampled_tiles' is not defined

In [19]:
# sampled_tiles.explore(
#     m=large_ecoregions.explore(
#        #m=gpd.read_file(ecoregions_path).cx[-179:179,45:75].explore(color='black'),
#        color='blue')
#     , color='red')

In [20]:
#visualize_samples(large_ecoregions, sampled_tiles, None, TO_CRS=3995)

## Run analysis: `carbon accumulation`

In [11]:
TILE_NUM_LIST = [3082, # no TCC trends raster
                 2996, # yes TCC trends raster but its all 255
                 3270, # yes TCC trends but only small portion of tile
                ]
TILE_NUM_LIST = [1897] #[1209] #[1558] #[2028] #[1209] #[2248]
TILE_NUM_LIST = list(range(2000,2006))
TILE_NUM_LIST = [177]

In [12]:
sorted(TILE_NUM_LIST)[0]

177

In [13]:
len(TILE_NUM_LIST)
print(TILE_NUM_LIST[-45:-40])

[]


In [14]:
import importlib
import carbon_accumulation
importlib.reload(carbon_accumulation)
importlib.reload(mosaiclib)

<module 'mosaiclib' from '/projects/code/icesat2_boreal/lib/mosaiclib.py'>

Note - added trimming of age raster to biomass raster to overcome different array shape (forest age was made with build_stack)

In [15]:
%%time

# MAINDIR = "/projects/carbon_accumulation_analysis_v2" # experimented with a handful of tiles across N. American ecoregions.
# MAINDIR = "/projects/carbon_accumulation_analysis_v2" # updated tile sampling to get tile list across many ecoregions (area_weighted, prefer_contained)
# MAINDIR = "/projects/carbon_accumulation_analysis_v3" # using v3.0 of AGB/HT 2020
MAINDIR = "/projects/my-public-bucket/local_output/carbon_accumulation_analysis_v4" # using v3.01_multiyr of AGB 2020

MAP_VERSION = '2020_v3.1_multiyr'

UPDATE_AGE = True

N_SIMS = 50

if UPDATE_AGE: 
    SIMDIR =  f"{MAINDIR}/N_SIMS_{N_SIMS}_update_age"
else:
    SIMDIR =  f"{MAINDIR}/N_SIMS_{N_SIMS}_no_update_age"
!mkdir -p $SIMDIR

#for TILE_NUM in sorted(TILE_NUM_LIST)[0:2]:
for TILE_NUM in TILE_NUM_LIST:
    
    DIR_TILE = f"{SIMDIR}/tile/{TILE_NUM:07}"
    TILE_NOTEBOOK = f'{DIR_TILE}/carbon_accumulation_plot_v2_{TILE_NUM:07}.ipynb'
    TILE_CACC_COG = f'{DIR_TILE}/boreal_cacc_2020_{TILE_NUM:07}.tif'
    
    if os.path.exists(TILE_CACC_COG) and os.path.exists(TILE_NOTEBOOK):
        print(f"Tile {TILE_NUM:07} complete.")
        continue
    
    if True:
        carbon_accumulation.CARBON_ACC_ANALYSIS(MAP_VERSION, TILE_NUM, num_simulations = N_SIMS, 
                                                N_PIX_SAMPLE=None, # creates a small pixel_dataframe , otherwise 10pct 
                                   random_seed = None, DO_WRITE_COG=True, UPDATE_AGE=UPDATE_AGE, local=True,
                                   extent_type = 'tile', output_dir = SIMDIR)

    # EXECUTE_NOTEBOOK = True
    # if get_cog_s3_path(TILE_NUM, mosaiclib.MISC_TINDEX_FN_DICT['TCC_TP_2020']) is not None:
    #     EXECUTE_NOTEBOOK = False
    # if not os.path.exists(TILE_CACC_COG):
    #     EXECUTE_NOTEBOOK = False
        
    # if EXECUTE_NOTEBOOK:
    #     pm.execute_notebook(
    #         #'/projects/code/maap_tools/notebooks/boreal_dynamics_plot_v2.ipynb',
    #         '/projects/code/maap_tools/notebooks/carbon_accumulation_plot_v2.ipynb',
    #         TILE_NOTEBOOK,
    #         parameters={
    #             'TILE_NUM ': TILE_NUM ,
    #             'OUTDIR': SIMDIR
    #         },
    #         kernel_name='ir'  # Specify the R kernel
    #     )
    # else:
    #     !rm -d $DIR_TILE

Starting carbon accumulation analysis...
Using map version: 2020_v3.1_multiyr and setting year as 2020.
Tile 177 not in s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/DECIDFRAC/build_stack_v2023_2/DECPRED_AB_2015/DECIDFRAC_tindex_master.csv
Reading and preparing raster data...
Read TP standage raster (index 8)...
Rasterizing ecoregion vector file...
Found 19 ecoregion polygons intersecting tile bounds
	Reprojected ecoregion gdf to CRS of template raster
Found 2 unique ecoregion names in this tile
After dissolving: 2 ecoregions
  1: Scandinavian Montane Birch forest and grasslands
  2: Scandinavian coastal conifer forests
Unique values in raster: [1 2]
Final ecoregion class labels: ['No Ecoregion', 'Scandinavian Montane Birch forest and grasslands', 'Scandinavian coastal conifer forests']
Biomass shape: (3000, 3000)
Age shape: (3000, 3000)
Trimming to: (3000, 3000)
	Trimmed:
		biomass_mean from (3000, 3000) to (3000, 3000)
		biomass_std from (3000, 3000) to (3000, 3000)
		age_m

Reading input: <open WarpedVRT name='WarpedVRT(/vsimem/00ac2aad-4e3a-4417-bc42-1810ea61407e/00ac2aad-4e3a-4417-bc42-1810ea61407e.tif)' mode='r'>



Writing img to memory...
Current stack shape:		(10,3000,3000)


Adding overviews...
Updating dataset tags...
Writing output to: /projects/my-public-bucket/local_output/carbon_accumulation_analysis_v4/N_SIMS_50_update_age/tile/0000177/boreal_cacc_2020_0000177.tif


Image written to disk:		 /projects/my-public-bucket/local_output/carbon_accumulation_analysis_v4/N_SIMS_50_update_age/tile/0000177/boreal_cacc_2020_0000177.tif
Dictionary of class rasters: dict_keys(['biomass_mean', 'biomass_std', 'meta', 'transform', 'crs', 'shape', 'age_mean', 'age_std', 'landcover', 'elevation', 'slope', 'tsri', 'tpi', 'slopemask', 'canopy_cover', 'canopy_trend', 'pvalue', 'deciduous', 'age_alt', 'ecoregions', 'ecoregion_class_labels', 'age_class', 'age_class_primary', 'age_mean_update', 'age_std_update', 'age_mean_primary', 'age_std_primary', 'trend_class', 'pvalue_class', 'deciduous_class'])

Trend class min/max: 5.0,5.0
Count of -9999: 0
Count of nan: 0
Count of unique: (array([5.], dtype=float32), array([9000000]))
Creating pixel-level dataframe - needed for complete summary csv files...
Total valid pixels: 60,924
Using all 60,924 valid pixels
Adding vector attributes...
  Processing ecoregions...
Added vector attributes. Final shape: (60924, 32)
Dataframe colum